
# Phase 11B-R — Multi-seed Robustness of Structured Stochastic Dynamics

This notebook repeats Phase 11B over multiple random seeds to test whether the main result is robust:

\[
dz_t =
\left[
-M_\theta\nabla U_\theta(z_t)
+
A_\theta\nabla H_\theta(z_t)
\right]dt
+
G_\theta(z_t)\,dW_t.
\]

For each seed we train:

1. a **Plain SDE** with unconstrained MLP drift;
2. a **Structured SDE** with dissipative + rotational drift;

using the same state-dependent diffusion architecture.

The notebook aggregates across seeds:

- drift MSE;
- dissipative-component MSE and cosine similarity;
- rotational-component MSE and cosine similarity;
- diffusion covariance error;
- held-out transition NLL;
- whitened-residual calibration;
- ensemble final-mean error;
- ensemble final-covariance error.

The user-provided seed `242431301` is included as the first seed.


In [ ]:

# Optional:
# !pip install -U torch numpy pandas matplotlib

import math
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32

# Yes: 242431301 is a perfectly valid random seed.
SEEDS = [
    242431301,
    20260927,
    314159265,
    271828182,
    161803398,
]

# Quick mode for a smoke test:
# SEEDS = [242431301, 20260927]

print("device:", DEVICE)
print("seeds:", SEEDS)



## 1. Shared ground-truth SDE

The ground-truth drift is the same as Phase 11B:

\[
b^\star(z)
=
-M^\star\nabla U^\star(z)
+
A^\star\nabla H^\star(z),
\]

with

\[
U^\star(x,y)
=
\frac12x^2+y^2+0.05x^4,
\]

\[
H^\star(x,y)
=
\frac12(x^2+y^2),
\]

\[
M^\star=
\begin{pmatrix}
1 & 0.2\\
0.2 & 0.7
\end{pmatrix},
\quad
A^\star=
\begin{pmatrix}
0 & -1.25\\
1.25 & 0
\end{pmatrix}.
\]

The diffusion is state dependent through

\[
D^\star(z)=G^\star(z)G^\star(z)^\top.
\]


In [ ]:

M_TRUE = torch.tensor(
    [[1.0, 0.20],
     [0.20, 0.70]],
    dtype=DTYPE,
    device=DEVICE,
)

A_TRUE = torch.tensor(
    [[0.0, -1.25],
     [1.25, 0.0]],
    dtype=DTYPE,
    device=DEVICE,
)

def true_grad_U(z):
    x = z[..., 0]
    y = z[..., 1]
    return torch.stack(
        [x + 0.20*x**3, 2.0*y],
        dim=-1,
    )

def true_grad_H(z):
    return z

def true_drift_components(z):
    gu = true_grad_U(z)
    gh = true_grad_H(z)
    fd = -(gu @ M_TRUE.T)
    fr = gh @ A_TRUE.T
    return fd, fr

def true_drift(z):
    fd, fr = true_drift_components(z)
    return fd + fr

def true_G(z):
    x = z[..., 0]
    y = z[..., 1]

    g11 = 0.18 + 0.08*torch.sigmoid(x)
    g21 = 0.05*torch.tanh(0.75*y)
    g22 = 0.13 + 0.07*torch.sigmoid(-y)

    zero = torch.zeros_like(g11)

    row1 = torch.stack([g11, zero], dim=-1)
    row2 = torch.stack([g21, g22], dim=-1)
    return torch.stack([row1, row2], dim=-2)

def true_D(z):
    Gm = true_G(z)
    return Gm @ Gm.transpose(-1, -2)

print("eig(M_true):", torch.linalg.eigvalsh(M_TRUE).detach().cpu().numpy())
print("skew error A_true:", torch.norm(A_TRUE + A_TRUE.T).item())



## 2. Dataset generation

Each seed generates a fresh training set and test set. This makes the robustness test stricter than merely reinitializing the network on identical data.


In [ ]:

DT = 0.04

N_TRAIN_PATHS = 320
N_TEST_PATHS = 80
STEPS_PER_PATH = 100

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def sample_initial_states(n):
    return -2.0 + 4.0*torch.rand(
        n, 2,
        device=DEVICE,
        dtype=DTYPE,
    )

@torch.no_grad()
def generate_transition_dataset(n_paths, steps, seed):
    set_seed(seed)

    z = sample_initial_states(n_paths)

    z_now_all = []
    z_next_all = []

    sqrt_dt = math.sqrt(DT)

    for _ in range(steps):
        b = true_drift(z)
        Gm = true_G(z)

        eps = torch.randn(
            n_paths, 2,
            device=DEVICE,
            dtype=DTYPE,
        )
        noise = torch.einsum("bij,bj->bi", Gm, eps)
        z_next = z + DT*b + sqrt_dt*noise

        z_now_all.append(z.clone())
        z_next_all.append(z_next.clone())
        z = z_next

    return {
        "z": torch.cat(z_now_all, dim=0),
        "z_next": torch.cat(z_next_all, dim=0),
    }



## 3. Model definitions

The architectures are unchanged from Phase 11B.


In [ ]:

class MLP(nn.Module):
    def __init__(self, din, dout, hidden=64, depth=2):
        super().__init__()
        layers = []
        d = din
        for _ in range(depth):
            layers += [nn.Linear(d, hidden), nn.Tanh()]
            d = hidden
        layers.append(nn.Linear(d, dout))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

class ScalarPotential(nn.Module):
    def __init__(self, hidden=64, depth=2):
        super().__init__()
        self.net = MLP(2, 1, hidden=hidden, depth=depth)

    def forward(self, z):
        return self.net(z).squeeze(-1)

def grad_scalar(model, z, create_graph=True):
    z_req = z.requires_grad_(True)
    s = model(z_req).sum()
    return torch.autograd.grad(
        s,
        z_req,
        create_graph=create_graph,
        retain_graph=create_graph,
    )[0]

class PSDMatrix2D(nn.Module):
    def __init__(self):
        super().__init__()
        self.raw = nn.Parameter(torch.tensor([0.2, 0.0, 0.2], dtype=DTYPE))

    def matrix(self):
        l11 = F.softplus(self.raw[0]) + 1e-3
        l21 = self.raw[1]
        l22 = F.softplus(self.raw[2]) + 1e-3

        L = torch.stack([
            torch.stack([l11, torch.zeros_like(l11)]),
            torch.stack([l21, l22]),
        ])

        eye = torch.eye(2, device=L.device, dtype=L.dtype)
        return L @ L.T + 1e-4*eye

class SkewMatrix2D(nn.Module):
    def __init__(self):
        super().__init__()
        self.a = nn.Parameter(torch.tensor(0.1, dtype=DTYPE))

    def matrix(self):
        zero = torch.zeros_like(self.a)
        return torch.stack([
            torch.stack([zero, -self.a]),
            torch.stack([self.a, zero]),
        ])

class DiffusionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = MLP(2, 3, hidden=64, depth=2)

    def factor(self, z):
        raw = self.net(z)

        g11 = F.softplus(raw[..., 0]) + 1e-3
        g21 = raw[..., 1]
        g22 = F.softplus(raw[..., 2]) + 1e-3
        zero = torch.zeros_like(g11)

        row1 = torch.stack([g11, zero], dim=-1)
        row2 = torch.stack([g21, g22], dim=-1)

        return torch.stack([row1, row2], dim=-2)

class PlainSDE(nn.Module):
    def __init__(self):
        super().__init__()
        self.drift_net = MLP(2, 2, hidden=96, depth=3)
        self.diffusion = DiffusionNet()

    def drift(self, z, create_graph=True):
        return self.drift_net(z)

    def diffusion_factor(self, z):
        return self.diffusion.factor(z)

class StructuredSDE(nn.Module):
    def __init__(self):
        super().__init__()
        self.U = ScalarPotential(hidden=64, depth=2)
        self.H = ScalarPotential(hidden=64, depth=2)
        self.M = PSDMatrix2D()
        self.A = SkewMatrix2D()
        self.diffusion = DiffusionNet()

    def components(self, z, create_graph=True):
        gu = grad_scalar(self.U, z, create_graph=create_graph)
        gh = grad_scalar(self.H, z, create_graph=create_graph)

        M = self.M.matrix()
        A = self.A.matrix()

        fd = -(gu @ M.T)
        fr = gh @ A.T
        return fd, fr, gu, gh, M, A

    def drift(self, z, create_graph=True):
        fd, fr, *_ = self.components(z, create_graph=create_graph)
        return fd + fr

    def diffusion_factor(self, z):
        return self.diffusion.factor(z)



## 4. Transition likelihood


In [ ]:

LOG_2PI = math.log(2.0*math.pi)

def transition_nll(model, z, z_next, create_graph=True):
    b = model.drift(z, create_graph=create_graph)
    Gm = model.diffusion_factor(z)

    residual = (z_next-z) - DT*b

    Sigma = DT*(Gm @ Gm.transpose(-1, -2))
    eye = torch.eye(2, device=z.device, dtype=z.dtype)
    Sigma = Sigma + 1e-6*eye

    chol = torch.linalg.cholesky(Sigma)

    sol = torch.cholesky_solve(
        residual.unsqueeze(-1),
        chol,
    ).squeeze(-1)

    maha = (residual*sol).sum(dim=-1)
    logdet = 2.0*torch.log(
        torch.diagonal(chol, dim1=-2, dim2=-1)
    ).sum(dim=-1)

    return 0.5*(maha + logdet + 2*LOG_2PI).mean()



## 5. Training configuration

On a faster GPU laptop, the default 1200 steps per model should be reasonable.

For a first test you can set:

```python
CFG = TrainConfig(epochs=400)
```

and then restore 1200 for the final robustness run.


In [ ]:

@dataclass
class TrainConfig:
    epochs: int = 1200
    batch_size: int = 512
    lr: float = 1.5e-3
    weight_decay: float = 1e-6
    print_every: int = 400

CFG = TrainConfig()

def train_sde(model, data, seed, cfg=CFG):
    set_seed(seed)

    model = model.to(DEVICE)

    opt = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.lr,
        weight_decay=cfg.weight_decay,
    )

    z_all = data["z"]
    zn_all = data["z_next"]
    n = len(z_all)

    hist = []

    for epoch in range(1, cfg.epochs+1):
        idx = torch.randint(
            0,
            n,
            (cfg.batch_size,),
            device=DEVICE,
        )

        z = z_all[idx].clone()
        zn = zn_all[idx]

        model.train()
        loss = transition_nll(
            model,
            z,
            zn,
            create_graph=True,
        )

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            10.0,
        )
        opt.step()

        hist.append(float(loss.detach().cpu()))

        if epoch == 1 or epoch % cfg.print_every == 0:
            print(
                f"seed={seed} "
                f"{model.__class__.__name__} "
                f"epoch={epoch:4d} "
                f"NLL={hist[-1]:.6f}"
            )

    return model, hist



## 6. Evaluation helpers


In [ ]:

def fresh_states(seed, n=4000):
    set_seed(seed)
    return -2.0 + 4.0*torch.rand(
        n, 2,
        device=DEVICE,
        dtype=DTYPE,
    )

def mse_tensor(a, b):
    return float(F.mse_loss(a, b).detach().cpu())

def cosine_mean(a, b, eps=1e-8):
    num = (a*b).sum(dim=-1)
    den = (
        torch.norm(a, dim=-1)
        * torch.norm(b, dim=-1)
        + eps
    )
    return float((num/den).mean().detach().cpu())

def covariance_relative_error(Dhat, Dtrue, eps=1e-8):
    err = torch.linalg.matrix_norm(
        Dhat-Dtrue,
        ord="fro",
        dim=(-2,-1),
    )
    base = torch.linalg.matrix_norm(
        Dtrue,
        ord="fro",
        dim=(-2,-1),
    )
    return float((err/(base+eps)).mean().detach().cpu())

def evaluate_field(model, seed, name):
    z_eval = fresh_states(seed+10_000, 4000)

    with torch.no_grad():
        b_true = true_drift(z_eval)
        fd_true, fr_true = true_drift_components(z_eval)
        D_true = true_D(z_eval)

    if isinstance(model, PlainSDE):
        model.eval()
        with torch.no_grad():
            b = model.drift(z_eval)
            Gm = model.diffusion_factor(z_eval)
            D = Gm @ Gm.transpose(-1,-2)

        return {
            "seed": seed,
            "model": name,
            "drift_mse": mse_tensor(b, b_true),
            "diffusion_D_mse": mse_tensor(D, D_true),
            "diffusion_D_relative_fro": covariance_relative_error(D, D_true),
        }

    model.eval()
    z = z_eval.clone().requires_grad_(True)

    fd, fr, gu, gh, M, A = model.components(
        z,
        create_graph=False,
    )
    b = fd + fr

    Gm = model.diffusion_factor(z)
    D = Gm @ Gm.transpose(-1,-2)

    diss_rate = (gu*fd).sum(dim=-1)
    rot_rate = (gh*fr).sum(dim=-1)

    return {
        "seed": seed,
        "model": name,
        "drift_mse": mse_tensor(b, b_true),
        "diss_component_mse": mse_tensor(fd, fd_true),
        "rot_component_mse": mse_tensor(fr, fr_true),
        "diss_component_cosine": cosine_mean(fd, fd_true),
        "rot_component_cosine": cosine_mean(fr, fr_true),
        "diffusion_D_mse": mse_tensor(D, D_true),
        "diffusion_D_relative_fro": covariance_relative_error(D, D_true),
        "M_min_eig": float(torch.linalg.eigvalsh(M).min().detach().cpu()),
        "A_skew_error": float(torch.norm(A+A.T).detach().cpu()),
        "dissipation_violation_fraction": float(
            (diss_rate > 1e-6).float().mean().detach().cpu()
        ),
        "mean_dU_dt_diss": float(diss_rate.mean().detach().cpu()),
        "mean_abs_dH_dt_rot": float(rot_rate.abs().mean().detach().cpu()),
        "learned_A_scalar": float(model.A.a.detach().cpu()),
    }

def whitened_residuals(model, data):
    model.eval()

    z = data["z"].clone()
    zn = data["z_next"]

    if isinstance(model, StructuredSDE):
        z = z.requires_grad_(True)
        b = model.drift(z, create_graph=False)
    else:
        with torch.no_grad():
            b = model.drift(z)

    with torch.no_grad():
        Gm = model.diffusion_factor(z)

    residual_scaled = (
        (zn-z.detach()) - DT*b.detach()
    ) / math.sqrt(DT)

    white = torch.linalg.solve_triangular(
        Gm.detach(),
        residual_scaled.unsqueeze(-1),
        upper=False,
    ).squeeze(-1)

    return white

def test_nll(model, data):
    vals = []
    B = 2048

    for start in range(0, len(data["z"]), B):
        z = data["z"][start:start+B].clone()
        zn = data["z_next"][start:start+B]

        if isinstance(model, StructuredSDE):
            z = z.requires_grad_(True)
            val = transition_nll(
                model,
                z,
                zn,
                create_graph=False,
            )
        else:
            with torch.no_grad():
                val = transition_nll(
                    model,
                    z,
                    zn,
                    create_graph=False,
                )

        vals.append(float(val.detach().cpu()))

    return float(np.mean(vals))

def evaluate_calibration(model, data, seed, name):
    white = whitened_residuals(
        model,
        data,
    ).detach().cpu().numpy()

    mean = white.mean(axis=0)
    cov = np.cov(white.T)

    return {
        "seed": seed,
        "model": name,
        "test_nll": test_nll(model, data),
        "white_mean_0": float(mean[0]),
        "white_mean_1": float(mean[1]),
        "white_cov_fro_from_I": float(
            np.linalg.norm(cov-np.eye(2), ord="fro")
        ),
        "white_mean_squared_norm": float(
            np.mean(np.sum(white**2, axis=1))
        ),
    }



## 7. Ensemble rollout metrics

For speed, the robustness run uses a moderate ensemble and three initial conditions.


In [ ]:

@torch.no_grad()
def true_sde_step(z, eps):
    b = true_drift(z)
    Gm = true_G(z)
    noise = torch.einsum("bij,bj->bi", Gm, eps)
    return z + DT*b + math.sqrt(DT)*noise

def learned_sde_step(model, z, eps):
    if isinstance(model, StructuredSDE):
        z_req = z.detach().requires_grad_(True)
        b = model.drift(
            z_req,
            create_graph=False,
        ).detach()

        with torch.no_grad():
            Gm = model.diffusion_factor(
                z_req.detach()
            )
    else:
        with torch.no_grad():
            b = model.drift(z)
            Gm = model.diffusion_factor(z)

    noise = torch.einsum("bij,bj->bi", Gm, eps)
    return z.detach() + DT*b + math.sqrt(DT)*noise

def simulate_ensemble(kind, z0, eps_seq, model=None):
    n = eps_seq.shape[1]
    z = z0[None,:].repeat(n,1).to(DEVICE)

    for t in range(len(eps_seq)):
        if kind == "true":
            z = true_sde_step(z, eps_seq[t])
        else:
            z = learned_sde_step(
                model,
                z,
                eps_seq[t],
            )

    return z.detach()

def empirical_mean_cov(z):
    x = z.detach().cpu().numpy()
    return x.mean(axis=0), np.cov(x.T)

def evaluate_ensemble(model, seed, name):
    set_seed(seed+20_000)

    N_ENSEMBLE = 256
    N_ROLLOUT_STEPS = 80

    eps_seq = torch.randn(
        N_ROLLOUT_STEPS,
        N_ENSEMBLE,
        2,
        device=DEVICE,
        dtype=DTYPE,
    )

    z0_list = [
        torch.tensor([1.6, 0.6], device=DEVICE, dtype=DTYPE),
        torch.tensor([-1.2, 1.4], device=DEVICE, dtype=DTYPE),
        torch.tensor([0.4, -1.5], device=DEVICE, dtype=DTYPE),
    ]

    mean_errors = []
    cov_errors = []

    for z0 in z0_list:
        final_true = simulate_ensemble(
            "true",
            z0,
            eps_seq,
        )

        final_model = simulate_ensemble(
            "learned",
            z0,
            eps_seq,
            model=model,
        )

        mu_true, cov_true = empirical_mean_cov(final_true)
        mu, cov = empirical_mean_cov(final_model)

        mean_errors.append(
            np.linalg.norm(mu-mu_true)
        )
        cov_errors.append(
            np.linalg.norm(
                cov-cov_true,
                ord="fro",
            )
        )

    return {
        "seed": seed,
        "model": name,
        "ensemble_mean_error": float(np.mean(mean_errors)),
        "ensemble_cov_fro_error": float(np.mean(cov_errors)),
    }



## 8. Run all seeds

This is the expensive cell.

Each seed trains two models, so with 5 seeds the notebook trains 10 models.


In [ ]:

all_field = []
all_calibration = []
all_ensemble = []
training_rows = []

for i, seed in enumerate(SEEDS, start=1):
    print("="*72)
    print(f"SEED {i}/{len(SEEDS)}: {seed}")
    print("="*72)

    train_data = generate_transition_dataset(
        N_TRAIN_PATHS,
        STEPS_PER_PATH,
        seed=seed+1,
    )

    test_data = generate_transition_dataset(
        N_TEST_PATHS,
        STEPS_PER_PATH,
        seed=seed+2,
    )

    set_seed(seed+100)
    plain = PlainSDE()

    set_seed(seed+200)
    structured = StructuredSDE()

    plain, hist_plain = train_sde(
        plain,
        train_data,
        seed=seed+1000,
    )

    structured, hist_struct = train_sde(
        structured,
        train_data,
        seed=seed+2000,
    )

    training_rows += [
        {
            "seed": seed,
            "model": "plain_sde",
            "final_train_nll": hist_plain[-1],
            "best_train_nll": min(hist_plain),
        },
        {
            "seed": seed,
            "model": "structured_sde",
            "final_train_nll": hist_struct[-1],
            "best_train_nll": min(hist_struct),
        },
    ]

    all_field += [
        evaluate_field(
            plain,
            seed,
            "plain_sde",
        ),
        evaluate_field(
            structured,
            seed,
            "structured_sde",
        ),
    ]

    all_calibration += [
        evaluate_calibration(
            plain,
            test_data,
            seed,
            "plain_sde",
        ),
        evaluate_calibration(
            structured,
            test_data,
            seed,
            "structured_sde",
        ),
    ]

    all_ensemble += [
        evaluate_ensemble(
            plain,
            seed,
            "plain_sde",
        ),
        evaluate_ensemble(
            structured,
            seed,
            "structured_sde",
        ),
    ]

field_df = pd.DataFrame(all_field)
calibration_df = pd.DataFrame(all_calibration)
ensemble_df = pd.DataFrame(all_ensemble)
training_df = pd.DataFrame(training_rows)

display(field_df)
display(calibration_df)
display(ensemble_df)



## 9. Aggregate across seeds

We report mean, standard deviation, median, minimum, and maximum.

For a robustness claim, pay particular attention to whether the structured-vs-plain advantage has the same sign across individual seeds.


In [ ]:

merged = (
    field_df
    .merge(
        calibration_df,
        on=["seed","model"],
        how="left",
    )
    .merge(
        ensemble_df,
        on=["seed","model"],
        how="left",
    )
    .merge(
        training_df,
        on=["seed","model"],
        how="left",
    )
)

display(merged)

metric_cols = [
    c for c in merged.columns
    if c not in {"seed","model"}
    and pd.api.types.is_numeric_dtype(merged[c])
]

summary_rows = []

for model, g in merged.groupby("model"):
    for metric in metric_cols:
        x = g[metric].dropna()

        if len(x) == 0:
            continue

        summary_rows.append({
            "model": model,
            "metric": metric,
            "n_seeds": len(x),
            "mean": x.mean(),
            "std": x.std(ddof=1) if len(x) > 1 else np.nan,
            "median": x.median(),
            "min": x.min(),
            "max": x.max(),
        })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)



## 10. Paired structured-vs-plain comparisons

Because both architectures are evaluated under the same outer seed, we can compare them seed by seed.

For error metrics, define

\[
\Delta
=
\text{structured}
-
\text{plain}.
\]

Thus:

\[
\Delta<0
\]

means the structured model is better.

We also report the fraction of seeds where structured wins.


In [ ]:

COMPARE_METRICS = [
    "drift_mse",
    "diffusion_D_relative_fro",
    "test_nll",
    "white_cov_fro_from_I",
    "ensemble_mean_error",
    "ensemble_cov_fro_error",
]

paired_rows = []

wide = merged.pivot(
    index="seed",
    columns="model",
)

for metric in COMPARE_METRICS:
    if (
        (metric, "plain_sde") not in wide.columns
        or
        (metric, "structured_sde") not in wide.columns
    ):
        continue

    p = wide[(metric, "plain_sde")]
    s = wide[(metric, "structured_sde")]

    delta = s-p

    paired_rows.append({
        "metric": metric,
        "mean_plain": float(p.mean()),
        "mean_structured": float(s.mean()),
        "mean_delta_struct_minus_plain": float(delta.mean()),
        "median_delta": float(delta.median()),
        "structured_win_fraction": float((delta < 0).mean()),
        "n_seeds": int(delta.notna().sum()),
    })

paired_df = pd.DataFrame(paired_rows)
display(paired_df)


In [ ]:

# Simple cross-seed plots.
plot_metrics = [
    "drift_mse",
    "diffusion_D_relative_fro",
    "test_nll",
    "ensemble_mean_error",
    "ensemble_cov_fro_error",
]

for metric in plot_metrics:
    plt.figure(figsize=(7,4))

    for model, g in merged.groupby("model"):
        g = g.sort_values("seed")
        plt.plot(
            [str(x) for x in g["seed"]],
            g[metric],
            marker="o",
            label=model,
        )

    plt.xlabel("seed")
    plt.ylabel(metric)
    plt.title(f"Phase 11B-R — {metric}")
    plt.legend()
    plt.xticks(rotation=30)
    plt.show()



## 11. Export


In [ ]:

field_df.to_csv(
    "phase11BR_field_metrics_by_seed.csv",
    index=False,
)

calibration_df.to_csv(
    "phase11BR_calibration_by_seed.csv",
    index=False,
)

ensemble_df.to_csv(
    "phase11BR_ensemble_by_seed.csv",
    index=False,
)

merged.to_csv(
    "phase11BR_all_metrics_by_seed.csv",
    index=False,
)

summary_df.to_csv(
    "phase11BR_summary_across_seeds.csv",
    index=False,
)

paired_df.to_csv(
    "phase11BR_paired_comparison.csv",
    index=False,
)

print("saved:")
print(" - phase11BR_field_metrics_by_seed.csv")
print(" - phase11BR_calibration_by_seed.csv")
print(" - phase11BR_ensemble_by_seed.csv")
print(" - phase11BR_all_metrics_by_seed.csv")
print(" - phase11BR_summary_across_seeds.csv")
print(" - phase11BR_paired_comparison.csv")



# Phase 11B-R decision gate

The Phase 11B result is robust enough to justify Phase 12 if the multi-seed run shows:

1. structured drift error lower than plain in most seeds;
2. dissipative and rotational cosine similarity consistently high;
3. structural identities remain satisfied for every seed;
4. structured ensemble-mean error is competitive or better in most seeds;
5. diffusion covariance error does not become unstable or catastrophically worse;
6. held-out NLL remains competitive;
7. no isolated seed dominates the conclusion.

A useful threshold is not a single arbitrary \(p\)-value at five seeds. At this stage we care more about:

\[
\text{effect consistency}
+
\text{effect size}
+
\text{variance across seeds}.
\]

If the pattern survives, Phase 12 can move from the synthetic SDE to a learned latent representation.
